In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import psignifit as pf
from tqdm import tqdm
from scipy.stats import gamma, lognorm, norm

from psychoanalyze import data
from datetime import datetime

pd.set_option('io.hdf.default_format','table')

In [15]:
points = data.load('points').drop(data.outliers['points'])
curves = data.load('curves')
# sessions = pd.read_csv('../data/1-normalized/sessions.csv', parse_dates=['Date'])
# monkeys = pd.read_csv('../data/1-normalized/subjects.csv', parse_dates=['Date of Implant Surgery'])

In [3]:
curve_columns = ['Monkey','Date','Ref Amp','Ref PW', 'Ref Freq','Ref Dur', 'Active Channels','Return Channels']
curve_calculated_columns = ['FAs','CRs','Return Channel(s)','Channel(s)','n_catch','X dimension',
                            'base', 'location','width','lambda','gamma','beta',
                            'location_CI_95', 'width_CI_95', 'lambda_CI_95', 'gamma_CI_95', 'beta_CI_95',
                            'location_CI_5', 'width_CI_5', 'lambda_CI_5', 'gamma_CI_5', 'beta_CI_5']
point_columns = ['Comp Amp','Comp PW']

In [4]:
curves = points.groupby(curve_columns)[['FAs','CRs']].sum().reset_index()

In [5]:
points = points.drop(columns=['FAs','CRs'])

In [6]:
sessions_monkeys = sessions.merge(monkeys[['Monkey', 'Date of Implant Surgery']])
sessions['Days'] = (sessions_monkeys['Date'] - sessions_monkeys['Date of Implant Surgery']).dt.days

In [7]:
curves = data.channelint2mask(curves)
curves = data.channelmask2num(curves)

In [8]:
points['n'] = points['Hits'] + points['Misses']
curves['n_catch'] = curves['FAs'] + curves['CRs']
curves['FA_rate'] = curves['FAs'] / curves['n_catch']
points['HR'] = points['Hits'] / points['n']

In [9]:
curves.loc[(curves['Ref Amp'] < 2) | (curves['Ref PW'] < 2), 'Experiment Type'] = 'Detection'
curves.loc[(curves['Ref Amp'] >= 2) & (curves['Ref PW'] >= 2), 'Experiment Type'] = 'Discrimination'

In [10]:
def find_x_dim(df):
    n_Amp = df['Comp Amp'].nunique()
    n_PW = df['Comp PW'].nunique()
    if n_Amp > n_PW:
        df['X Dimension'] = 'Amp'
    elif n_Amp < n_PW:
        df['X Dimension'] = 'PW'
    elif (n_Amp < 3) & (n_PW < 3):
        df['X Dimension'] = 'None'
    else:
        df['X Dimension'] = 'Both'
    return df[curve_columns + ['X Dimension']]
    
curves_dim = points.groupby(curve_columns).apply(find_x_dim).drop_duplicates()
curves = curves.merge(curves_dim)

In [11]:
curves

,Monkey,Date,Ref Amp,Ref PW,Ref Freq,Ref Dur,Active Channels,Return Channels,FAs,CRs,Act Chan Mask,Ret Chan Mask,Return Channel(s),Channel(s),n_catch,FA_rate,Experiment Type,X Dimension
0,U,20160926,0,0,0,0,8,128,8,189,00001000,10000000,1,5,197,0.040609,Detection,Amp
1,U,20160927,0,0,0,0,9,144,37,191,00001001,10010000,1+4,5+8,228,0.162281,Detection,Amp
2,U,20160928,0,0,0,0,8,128,37,201,00001000,10000000,1,5,238,0.155462,Detection,Amp
3,U,20160929,0,0,0,0,15,128,83,246,00001111,10000000,1,5+6+7+8,329,0.252280,Detection,Amp
4,U,20161003,0,0,0,0,15,64,11,295,00001111,01000000,2,5+6+7+8,306,0.035948,Detection,Amp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1076,Z,20170120,0,0,0,0,15,0,0,0,00001111,00000000,,5+6+7+8,0,NaN,Detection,None
1077,Z,20170120,0,0,0,0,15,16,21,113,00001111,00010000,4,5+6+7+8,134,0.156716,Detection,Amp
1078,Z,20170120,0,0,0,0,15,32,18,339,00001111,00100000,3,5+6+7+8,357,0.050420,Detection,Amp
1079,Z,20170120,0,0,0,0,15,64,5,129,00001111,01000000,2,5+6+7+8,134,0.037313,Detection,Amp


In [12]:
curves = curves[curves['X Dimension'].isin(['Amp','PW'])]

In [6]:
curves_points = curves.merge(points, left_index=True, right_index=True)

In [21]:
curves_points['x'] = curves_points['Comp PW'].where(curves_points['X Dimension'] == 'PW', other=curves_points['Comp Amp'])
curves_points['base'] = curves_points['Ref Amp'].where(curves_points['X Dimension'] == 'PW', other=curves_points['Ref PW'])

In [22]:
points_x = curves_points[curve_columns + point_columns + ['x','base']]
points = points.merge(points_x)
# points.to_csv('../data/2-calculated/points.csv', index=False)

In [25]:
points

,Monkey,Date,Ref Amp,Ref PW,Ref Freq,Ref Dur,Active Channels,Return Channels,Comp Amp,Comp PW,Misses,Hits,n,x,base
0,U,20160926,0,0,0,0,8,128,1,1,0,0,0,1.0,1.0
1,U,20160926,0,0,0,0,8,128,75,200,0,1,1,75.0,200.0
2,U,20160926,0,0,0,0,8,128,350,200,1,1,2,350.0,200.0
3,U,20160926,0,0,0,0,8,128,492,200,0,1,1,492.0,200.0
4,U,20160926,0,0,0,0,8,128,1000,200,3,6,9,1000.0,200.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9457,Z,20170120,0,0,0,0,15,128,90,200,0,10,10,90.0,200.0
9458,Z,20170120,0,0,0,0,15,128,110,200,0,12,12,110.0,200.0
9459,Z,20170120,0,0,0,0,15,128,130,200,0,12,12,130.0,200.0
9460,Z,20170120,0,0,0,0,15,128,150,200,0,11,11,150.0,200.0


In [27]:
points = points[points['n'] > 3]
# points.to_csv('../data/2-calculated/points.csv', index=False)

In [28]:
curves_points = points.merge(curves)
curves_points

,Monkey,Date,Ref Amp,Ref PW,Ref Freq,Ref Dur,Active Channels,Return Channels,Comp Amp,Comp PW,...,base,FAs,CRs,Act Chan Mask,Ret Chan Mask,Return Channel(s),Channel(s),n_catch,FA_rate,X dimension
0,U,20160926,0,0,0,0,8,128,1000,200,...,200.0,8,189,00001000,10000000,1,5,197,0.040609,Amp
1,U,20160926,0,0,0,0,8,128,1071,200,...,200.0,8,189,00001000,10000000,1,5,197,0.040609,Amp
2,U,20160926,0,0,0,0,8,128,1142,200,...,200.0,8,189,00001000,10000000,1,5,197,0.040609,Amp
3,U,20160926,0,0,0,0,8,128,1214,200,...,200.0,8,189,00001000,10000000,1,5,197,0.040609,Amp
4,U,20160926,0,0,0,0,8,128,1285,200,...,200.0,8,189,00001000,10000000,1,5,197,0.040609,Amp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8068,Z,20170120,0,0,0,0,15,128,90,200,...,200.0,15,222,00001111,10000000,1,5+6+7+8,237,0.063291,Amp
8069,Z,20170120,0,0,0,0,15,128,110,200,...,200.0,15,222,00001111,10000000,1,5+6+7+8,237,0.063291,Amp
8070,Z,20170120,0,0,0,0,15,128,130,200,...,200.0,15,222,00001111,10000000,1,5+6+7+8,237,0.063291,Amp
8071,Z,20170120,0,0,0,0,15,128,150,200,...,200.0,15,222,00001111,10000000,1,5+6+7+8,237,0.063291,Amp


In [30]:
tqdm.pandas()
fitted_curves = curves_points.groupby(curve_columns + ['base']).progress_apply(fit_curve)

  0%|          | 0/993 [00:00<?, ?it/s]

(array([], dtype=int64),)


  0%|          | 2/993 [00:39<5:22:42, 19.54s/it]

(array([], dtype=int64),)


  0%|          | 3/993 [00:58<5:19:49, 19.38s/it]

...

100%|█████████▉| 992/993 [5:15:21<00:24, 24.73s/it]

(array([], dtype=int64),)


100%|██████████| 993/993 [5:15:51<00:00, 26.09s/it]

(array([], dtype=int64),)


100%|██████████| 993/993 [5:16:18<00:00, 19.11s/it]


In [30]:
curve_990_old = curves_points[curves_points['id']==990]

Monkey  Date        Ref Amp  Ref PW  Ref Freq  Ref Dur  Active Channels  Return Channels  Comp PW  Comp Amp
Z       2017-01-20  0        0       0         0        15               32               200      10          132.868287
                                                                                                   20          132.868287
                                                                                                   30          132.868287
                                                                                                   40          132.868287
                                                                                                   50          132.868287
                                                                                                   60          132.868287
                                                                                                   70          132.868287
                                      

In [20]:
curve_990 = curves[curves['id'] == 990]
curve_990 = curve_990.merge(points, left_index=True, right_index=True)

In [21]:
curve_990_new = data.fit_curve(curve_990)

(array([], dtype=int64),)


/home/tyler/.cache/pypoetry/virtualenvs/psychoanalyze-Jj9wTL_R-py3.8/lib/python3.8/site-packages/psignifit/likelihood.py:191: RuntimeWarning: divide by zero encountered in log
  pbin = pbin  + (ni-ki)*np.log(1-psi)


In [48]:
df = curve_990_new.reset_index(level=[-1, -2])
df = df[~df.index.duplicated()]

In [48]:
curves.loc[curves['id'] == 990, 'location'] = df['location']
curves.loc[curves['id'] == 990, 'width'] = df['width']
curves.loc[curves['id'] == 990, 'gamma'] = df['gamma']
curves.loc[curves['id'] == 990, 'lambda'] = df['lambda']
curves.loc[curves['id'] == 990, 'location_CI_95'] = df['location_CI_95']
curves.loc[curves['id'] == 990, 'width_CI_95'] = df['width_CI_95']
curves.loc[curves['id'] == 990, 'gamma_CI_95'] = df['gamma_CI_95']
curves.loc[curves['id'] == 990, 'lambda_CI_95'] = df['lambda_CI_95']
curves[curves['id'] == 990]

KeyError: ('Z', '2017-01-20', 0, 0, 0, 0, 15, 32)

In [31]:
fitted_curves_copy = fitted_curves
fitted_curves_copy.drop_duplicates(subset=curve_columns + ['base'], inplace=True)

In [34]:
fitted_curves_copy[curve_columns + curve_calculated_columns]

,Monkey,Date,Ref Amp,Ref PW,Ref Freq,Ref Dur,Active Channels,Return Channels,FAs,CRs,...,location_CI_95,width_CI_95,lambda_CI_95,gamma_CI_95,beta_CI_95,location_CI_5,width_CI_5,lambda_CI_5,gamma_CI_5,beta_CI_5
0,U,20160926,0,0,0,0,8,128,8,189,...,786.382167,507.116504,0.004255,0.005121,0.006965,1274.764314,2461.477576,0.199721,0.266349,0.368554
20,U,20160927,0,0,0,0,9,144,37,191,...,850.784753,563.004764,0.009054,0.029938,0.001704,1339.725481,1852.611993,0.265826,0.393491,0.170536
28,U,20160928,0,0,0,0,8,128,37,201,...,819.201226,501.302056,0.015430,0.007456,0.001905,1081.096471,1573.931481,0.155874,0.234235,0.201022
37,U,20160929,0,0,0,0,15,128,83,246,...,811.465608,628.348773,0.001393,0.045894,0.002259,1093.323336,1621.918867,0.098186,0.299524,0.221197
46,U,20161003,0,0,0,0,15,64,11,295,...,770.876373,392.991960,0.000444,0.001699,0.002142,906.398174,883.176659,0.043664,0.121868,0.251619
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8020,Z,20170119,0,0,0,0,15,128,34,476,...,22.097915,12.407337,0.009308,0.000906,0.011587,29.691813,32.254696,0.066511,0.138034,0.524543
8028,Z,20170120,0,0,0,0,15,16,21,113,...,57.784375,13.705709,0.010574,0.031409,0.003131,72.277978,67.303253,0.111637,0.321256,0.336625
8036,Z,20170120,0,0,0,0,15,32,18,339,...,56.574764,17.435473,0.003409,0.017295,0.449952,162.743213,282.620962,0.407024,0.281223,0.708815
8048,Z,20170120,0,0,0,0,15,64,5,129,...,39.339222,50.805831,0.001000,0.002636,0.005073,66.909269,147.123905,0.132374,0.202644,0.455702


In [35]:
fitted_curves_copy.to_csv('../data/2-calculated/curves2.csv', index=False)

In [57]:
curves = data.add_relative_errors(curves)
# curves['Channel(s)'], curves['Return Channel(s)'] = curves['Return Channel(s)'], curves['Channel(s)'].copy()


In [60]:
curves.to_hdf('../data/2-calculated.h5', 'curves')
# curves.to_csv('../data/2-calculated/curves2.csv', index=False)

/home/tyler/.cache/pypoetry/virtualenvs/psychoanalyze-Jj9wTL_R-py3.8/lib/python3.8/site-packages/tables/path.py:155: NaturalNameWarning: object name is not a valid Python identifier: 'Return Channels'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/home/tyler/.cache/pypoetry/virtualenvs/psychoanalyze-Jj9wTL_R-py3.8/lib/python3.8/site-packages/tables/path.py:155: NaturalNameWarning: object name is not a valid Python identifier: 'Active Channels'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/home/tyler/.cache/pypoetry/virtualenvs/psychoanalyze-Jj9wTL_R-py3.8/lib/python3.8/site-packages/tables/path.py:155: NaturalNameWarning: object name is not a valid Python identifier: 'Ref Dur'; it d

In [12]:
distal_group = [
    '1','2','3','4','1+2','2+3','3+4','1+4',
    '1+2+3','2+3+4','1+3+4','1+2+4', 
    '1+2+3+4',
]
proximal_group = [
    '5','6','7','8','5+6','6+7','7+8','5+8',
    '5+6+7','6+7+8','5+7+8','5+6+8','5+6+7+8'
]

In [13]:
curves.loc[curves['Channel(s)'].isin(distal_group) & curves['Return Channel(s)'].isin(distal_group), 'Electrode Config'] = 'Multipolar'
curves.loc[curves['Channel(s)'].isin(proximal_group) & curves['Return Channel(s)'].isin(proximal_group), 'Electrode Config'] = 'Multipolar'
curves.loc[curves['Channel(s)'].isin(proximal_group) & curves['Return Channel(s)'].isin(distal_group), 'Electrode Config'] = 'Monopolar'
curves.loc[curves['Channel(s)'].isin(distal_group) & curves['Return Channel(s)'].isin(proximal_group), 'Electrode Config'] = 'Monopolar'


In [14]:
curves['Electrode Config'].value_counts()

Monopolar     890
Multipolar     99
Name: Electrode Config, dtype: int64

In [ ]:
min_max = curves_points.groupby(curves.index.names)['x'].agg(['min', 'max'])
curves = curves.merge(min_max, left_index=True, right_index=True)

In [15]:
curves.to_csv('../data/2-calculated/curves2.csv', index=False)

In [16]:
impedances = pd.read_csv('../data/0-raw/impedances.csv', parse_dates=['Date'])

,Date,Monkey,1,2,3,4,5,6,7,8
0,2016-04-28,Y,NaN,5600.000000,6000.00000,9100.00000,7600.00000,6900.00000,6300.0000,9900.00000
1,2016-05-25,Y,7200.000,6900.000000,7900.00000,9100.00000,7200.00000,7900.00000,6800.0000,7900.00000
2,2016-07-06,Y,10500.000,9900.000000,9900.00000,11700.00000,9200.00000,9500.00000,8100.0000,11500.00000
3,2016-07-26,U,13100.000,11400.000000,16600.00000,12600.00000,13500.00000,11500.00000,15500.0000,15100.00000
4,2018-06-13,U,NaN,22508.300000,6702.55600,9556.43800,5271.66200,6580.15880,3365.4093,3239.66000
5,2017-11-13,Y,NaN,9644.653394,12150.25543,11547.22929,13951.46332,20559.57961,149280.0228,41169.91672
6,2018-01-11,Y,NaN,100745.100000,6948.06600,17257.27000,9591.58400,8436.94900,62547.9900,23621.70000
7,2017-12-13,Y,5534.753,NaN,6152.94100,17821.50000,11496.50000,9295.20300,71058.9100,97559.07000
8,2017-02-11,U,4820.000,12895.000000,6108.00000,6392.00000,8466.00000,5977.00000,6537.0000,6564.00000
9,2016-12-13,U,7514.000,11215.690000,15394.12000,13653.45000,6392.52200,11000.60000,19097.1300,9136.84700


In [21]:
impedances = impedances.merge(monkeys[['Monkey', 'Date of Implant Surgery']])
impedances['Days'] = (impedances['Date'] - impedances['Date of Implant Surgery']).dt.days
impedances = impedances.drop(columns='Date of Implant Surgery')

In [22]:
impedances

,Monkey,Date,Channel,1 KHz Impedance,Days
0,Y,2016-04-28,1,NaN,48
1,Y,2016-05-25,1,7200.0,75
2,Y,2016-07-06,1,10500.0,117
3,Y,2017-11-13,1,NaN,612
4,Y,2018-01-11,1,NaN,671
...,...,...,...,...,...
131,Z,2016-12-12,7,17212.0,87
132,Z,2016-09-15,7,2403.0,-1
133,Z,2016-10-21,8,9982.0,35
134,Z,2016-12-12,8,17385.0,87


In [23]:
impedances.to_csv('../data/2-calculated/impedances.csv', index=False)

In [28]:
pd.read_csv('../data/2-calculated/curves2.csv').drop(columns='Unnamed: 0').to_csv('../data/2-calculated/curves2.csv', index=False)

In [6]:
points = pd.read_hdf('../data/2-calculated.h5', 'points')
points['Ref Current'] = points.index.get_level_values('Ref Amp')*points.index.get_level_values['Ref PW']
points['Comp Current'] = points.index.get_level_values['Comp Amp']*points.index.get_level_values['Comp PW']

TypeError: 'method' object is not subscriptable

In [23]:
curves = data.load('curves', outliers=None)

In [24]:
curves['Ref Current'] = curves.index.get_level_values('Ref Amp') * curves.index.get_level_values('Ref PW') / 1000
curves['Ref Current']

Monkey  Date        Ref Amp  Ref PW  Ref Freq  Ref Dur  Active Channels  Return Channels
U       2016-09-27  0        0       0         0        9                144                0.0
        2016-09-28  0        0       0         0        8                128                0.0
        2016-09-29  0        0       0         0        15               128                0.0
        2016-10-03  0        0       0         0        15               64                 0.0
        2016-10-05  0        0       0         0        240              4                  0.0
                                                                                           ... 
Z       2017-01-19  0        0       0         0        15               128                0.0
        2017-01-20  0        0       0         0        15               16                 0.0
                                                                         32                 0.0
                                               

In [25]:
curves.to_hdf('../data/2-calculated.h5', 'curves')

In [18]:
points = data.load('points', outliers=None)

In [19]:
points['Comp Current'] = points.index.get_level_values('Comp Amp') * points.index.get_level_values('Comp PW') / 1000
points.to_hdf('../data/2-calculated.h5', 'points')

In [ ]:
points.to_hdf('../data/1-normalized.h5', 'points')
sessions.to_hdf('../data/1-normalized.h5', 'sessions')
monkeys.to_hdf('../data/1-normalized.h5', 'monkeys')